# 1 Fusion par pyramide laplacienne

In [4]:
import cv2
import numpy as np

def build_laplacian_pyramid(image, levels):
    Im = image.copy()
    GP = [Im]
    for i in range(levels-1):
        Im = cv2.pyrDown(Im)
        GP.append(Im)

    LP = []
    for i in range(len(GP)-1):
        L = GP[i] - cv2.pyrUp(GP[i+1])
        LP.append(L)

    LP.append(GP[-1])
    return LP

def max_lap_pyr(lp1, lp2):

    LPmax = []
    n = len(lp1)

    for i in range(n-1):
        
        L1 = lp1[i]
        L2 = lp2[i]
        mask = np.sqrt(np.sum(L1**2,2)) > np.sqrt(np.sum(L2**2,2))
        mask = mask[:,:,np.newaxis]

        max = np.where(mask,L1,L2)
        LPmax.append(max)
    
    return LPmax

def fus_last_lvl(Img1, Img2):
    return cv2.addWeighted(Img1, 0.5, Img2, 0.5, 0)


def fus_lap_pyr(img1, img2, levels):
    lp1 = build_laplacian_pyramid(img1, levels)
    lp2 = build_laplacian_pyramid(img2, levels)

    LPmax = max_lap_pyr(lp1, lp2)

    img_fus = fus_last_lvl(lp1[-1], lp2[-1])

    for i in range(levels - 2, -1, -1):
        img_fus = cv2.pyrUp(img_fus)
        img_fus = img_fus + LPmax[i]


    return img_fus


Test

In [24]:

img1 = cv2.imread("test_fusion_2_images/set1/test11_claire.png") 
img2 = cv2.imread("test_fusion_2_images/set1/test11_sombre.png")
img3 = cv2.imread("test_fusion_2_images/set2/test11.png") 
img4 = cv2.imread("test_fusion_2_images/set2/test11_flou_leger.png")


for j in range(4,7):
    img_fus1 = fus_lap_pyr(img1, img2, j)
    img_fus1 = np.clip(img_fus1, 0, 255).astype(np.uint8)

    img_fus2 = fus_lap_pyr(img3, img4, j)
    img_fus2 = np.clip(img_fus2, 0, 255).astype(np.uint8)

    cv2.imshow(f"Test set 1, niveau {j}", img_fus1)
    cv2.imshow(f"Test set 2, niveau {j}", img_fus2)



cv2.waitKey(0)
cv2.destroyAllWindows()



# 2 Fusion Multi-Images

In [25]:
def fusLap_N_Images(ImageList,NumberImages, Level):
    Pyrlist = []
    for img in ImageList:
        Pyrlist.append(build_laplacian_pyramid(img,Level))

    max = max_lap_pyr(Pyrlist[0],Pyrlist[1])

    for i in range(2,NumberImages):
        max = max_lap_pyr(Pyrlist[i],max)

    Gaulist = []
    for  i in range(NumberImages):
        Gaulist.append(Pyrlist[i][-1])

    img_fus = np.mean(Gaulist,0).astype(np.uint8)

    for i in range(Level - 2, -1, -1):
        img_fus = cv2.pyrUp(img_fus)
        img_fus = img_fus + max[i]

        
    return img_fus
    

In [26]:

img1 = cv2.imread("test_fusion_N_images/set1N/test11_claire.png") 
img2 = cv2.imread("test_fusion_N_images/set1N/test11_sombre.png")
img3 = cv2.imread("test_fusion_N_images/set1N/test11.png")

img4 = cv2.imread("test_fusion_N_images/set2N/test11_flou_prononce.png") 
img5 = cv2.imread("test_fusion_N_images/set2N/test11_flou_leger.png")
img6 = cv2.imread("test_fusion_N_images/set2N/test11.png") 

list1 = [img1,img2,img3]
list2 = [img4,img5,img6]

for j in range(4,7):
    img_fus1 = fusLap_N_Images(list1,3,j)
    img_fus1 = np.clip(img_fus1, 0, 255).astype(np.uint8)

    img_fus2 = fusLap_N_Images(list2,3, j)
    img_fus2 = np.clip(img_fus2, 0, 255).astype(np.uint8)

    cv2.imshow(f"Test set 1, niveau {j}", img_fus1)
    cv2.imshow(f"Test set 2, niveau {j}", img_fus2)

cv2.waitKey(0)
cv2.destroyAllWindows()

